# Convert audit_v11 Timestamped Actions to Simple Left/Right/Camera/Other GT

This notebook programmatically converts the timestamped action annotations produced by `audit_timestamped_tool_interface.ipynb` into a compact per-clip format. It does not manually encode clip annotations; it reads `audit_v11` and the same interface taxonomy/constants used by the audit UI.

In [1]:

import csv
import json
import os
import sys
from pathlib import Path

ROOT_DIR = Path('/shared_data0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.audit_timestamped_tool_interface import (
    CAMERA_ACTION_CODES,
    EXTRA_ACTION_CODES,
    EXTRA_TOOL_TYPES,
    RETRACTION_DIRECTION_OPTIONS,
)

ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'audit_v11'
TAXONOMY_PATH = ANNOTATION_ROOT / 'taxonomy_v10.json'
OUT_DIR = ROOT_DIR / 'notebooks/artifacts/audit_v11_simple_action_gt'
SIMPLE_GT_PATH = OUT_DIR / 'audit_v11_simple_actions.json'
NATURAL_GT_PATH = OUT_DIR / 'audit_v11_simple_actions_natural_language.json'
OPTIONS_PATH = OUT_DIR / 'audit_v11_simple_action_options.json'
NATURAL_OPTIONS_PATH = OUT_DIR / 'audit_v11_simple_action_options_natural_language.json'
TRANSLATION_PAIRS_PATH = OUT_DIR / 'audit_v11_code_translation_pairs.json'
CSV_PATH = OUT_DIR / 'audit_v11_simple_actions_flat.csv'
NATURAL_CSV_PATH = OUT_DIR / 'audit_v11_simple_actions_natural_language_flat.csv'

OUT_DIR.mkdir(parents=True, exist_ok=True)
print('audit_v11:', AUDIT_V11_DIR)
print('taxonomy:', TAXONOMY_PATH)
print('out:', OUT_DIR)


audit_v11: /shared_data0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/audit_v11
taxonomy: /shared_data0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/taxonomy_v10.json
out: /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt


In [2]:

def read_json(path):
    return json.loads(Path(path).read_text())


def write_json(path, obj):
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(json.dumps(obj, indent=2) + '\n')


def taxonomy_options():
    taxonomy = read_json(TAXONOMY_PATH)
    options = {field: [item['value'] for item in items] for field, items in taxonomy['fields'].items()}
    for code in EXTRA_ACTION_CODES:
        if code not in options['action_code']:
            options['action_code'].append(code)
    for tool in EXTRA_TOOL_TYPES:
        if tool not in options['tool_type']:
            options['tool_type'].append(tool)
    return options


def is_set(value):
    return value not in (None, '', '(not set)', 'Null', 'null')


def frame_or_none(value):
    if value is None:
        return None
    try:
        return int(value)
    except Exception:
        return None


def sorted_segments(rows):
    return sorted(rows, key=lambda row: (row.get('start_frame') is None, row.get('start_frame') or -1, row.get('end_frame') or -1, json.dumps(row, sort_keys=True)))


def load_audit_v11_records():
    records = []
    for path in sorted(AUDIT_V11_DIR.glob('*.json')):
        for record in read_json(path):
            record = dict(record)
            record['_source_path'] = str(path)
            records.append(record)
    return records

records = load_audit_v11_records()
options = taxonomy_options()
print('records:', len(records))
print('taxonomy action codes:', len(options['action_code']))


records: 90
taxonomy action codes: 25


In [3]:
import re


def generic_other_segment(action):
    start = action.get('action_start_frame', action.get('start_frame'))
    end = action.get('action_end_frame', action.get('end_frame'))
    return {
        'start_frame': frame_or_none(start),
        'end_frame': frame_or_none(end),
        'target_structure': action.get('target_structure', '(not set)'),
        'target_context_1': action.get('target_context_1', '(not set)'),
        'target_context_2': action.get('target_context_2', '(not set)'),
        'actor_role': action.get('actor_role', '(not set)'),
        'tool_type': action.get('tool_type', '(not set)'),
        'action_code': action.get('action_code', '(not set)'),
        'description': other_segment_description(action),
        'rank': action.get('rank'),
    }


LEFT_CODE_TO_DIRECTION_FIELDS = {
    'KEEP_RETRACT_LATERAL': ('no', 'lateral', 'lateral'),
    'KEEP_RETRACT_MEDIAL': ('no', 'medial', 'medial'),
    'KEEP_RETRACT_UPWARD': ('no', 'upward', 'upward'),
    'RETRACT_LATERAL': ('yes', 'not_retracted', 'lateral'),
    'RETRACT_MEDIAL': ('yes', 'not_retracted', 'medial'),
    'RETRACT_LATERAL_TO_MEDIAL': ('yes', 'lateral', 'medial'),
    'RETRACT_LATERAL_TO_UPWARD': ('yes', 'lateral', 'upward'),
    'RETRACT_MEDIAL_TO_LATERAL': ('yes', 'medial', 'lateral'),
    'RETRACT_UPWARD_TO_LATERAL': ('yes', 'upward', 'lateral'),
}


ACTION_VERBS = {
    'DISSECT': 'dissects around',
    'RETRACT': 'retracts',
    'ASPIRATE': 'aspirates',
    'IRRIGATE': 'irrigates',
    'CLIP': 'clips',
    'CUT': 'cuts',
    'GRASP': 'grasps',
    'ICG_SWITCH': 'switches ICG',
}

CAMERA_DESCRIPTIONS = {
    'CAMERA_UNCERTAIN': 'The camera is uncertain.',
    'CAMERA_ZOOM_OUT': 'The camera zooms out.',
    'CAMERA_ZOOM_IN': 'The camera zooms in.',
    'CAMERA_REPOSITION': 'The camera repositions.',
    'CAMERA_NO_CHANGE': 'The camera does not change.',
}

NATURAL_CODE_TRANSLATIONS = {
    'KEEP_RETRACT_LATERAL': 'keep retracted lateral',
    'KEEP_RETRACT_MEDIAL': 'keep retracted medial',
    'KEEP_RETRACT_UPWARD': 'keep retracted upward',
    'RETRACT_LATERAL': 'retract to lateral',
    'RETRACT_MEDIAL': 'retract to medial',
    'RETRACT_LATERAL_TO_MEDIAL': 'retract from lateral to medial',
    'RETRACT_LATERAL_TO_UPWARD': 'retract from lateral to upward',
    'RETRACT_MEDIAL_TO_LATERAL': 'retract from medial to lateral',
    'RETRACT_UPWARD_TO_LATERAL': 'retract from upward to lateral',
    'CAMERA_UNCERTAIN': 'The camera is uncertain',
    'CAMERA_ZOOM_OUT': 'The camera zooms out',
    'CAMERA_ZOOM_IN': 'The camera zooms in',
    'CAMERA_REPOSITION': 'The camera repositions',
    'CAMERA_NO_CHANGE': 'The camera does not change',
    'DISSECT': 'dissect',
    'ICG_SWITCH': 'switch ICG',
}


def left_direction_fields(seg):
    code = seg.get('action_code', '(not set)')
    derived = LEFT_CODE_TO_DIRECTION_FIELDS.get(code, ('unsure', 'unsure', 'unsure'))
    changed = seg.get('changed')
    start_direction = seg.get('start_direction')
    end_direction = seg.get('end_direction')
    if changed in (None, '', 'unsure'):
        changed = derived[0]
    if start_direction in (None, '', 'unsure'):
        start_direction = derived[1]
    if end_direction in (None, '', 'unsure'):
        end_direction = derived[2]
    return changed, start_direction, end_direction


def humanize_code(value):
    text = str(value or '').strip()
    if not is_set(text):
        return ''
    if text in NATURAL_CODE_TRANSLATIONS:
        return NATURAL_CODE_TRANSLATIONS[text]
    text = re.sub(r'(?<=[a-z])(?=[A-Z])', ' ', text)
    return text.replace('_', ' ').replace('-', ' ').lower()


def complete_sentence(text):
    out = str(text or '').strip()
    if not out:
        return out
    return out if out.endswith(('.', '!', '?')) else out + '.'


def left_segment_description(seg):
    if is_set(seg.get('description')):
        return str(seg.get('description'))
    changed = seg.get('changed')
    start_direction = humanize_code(seg.get('start_direction'))
    end_direction = humanize_code(seg.get('end_direction'))
    if changed == 'no' and end_direction:
        return f'The grasper keeps the gallbladder neck retracted {end_direction}.'
    if start_direction and end_direction:
        return f'The grasper retracts the gallbladder neck from {start_direction} to {end_direction}.'
    code = humanize_code(seg.get('action_code'))
    return f'The grasper retracts the gallbladder neck with action {code}.' if code else 'The grasper retracts the gallbladder neck.'


def right_segment_description(seg):
    if is_set(seg.get('description')):
        return str(seg.get('description'))
    tool = humanize_code(seg.get('tool_type')) or 'right instrument'
    action_code = seg.get('action_code')
    verb = ACTION_VERBS.get(action_code, humanize_code(action_code) or 'acts')
    target = humanize_code(seg.get('target_structure'))
    contexts = [humanize_code(seg.get('target_context_1')), humanize_code(seg.get('target_context_2'))]
    contexts = [context for context in contexts if context and context != 'not set']
    target_phrase = f' the {target}' if target else ''
    if contexts:
        return f'The {tool} {verb}{target_phrase}, {", and ".join(contexts)}.'
    return f'The {tool} {verb}{target_phrase}.'


def camera_segment_description(seg):
    if is_set(seg.get('description')):
        return str(seg.get('description'))
    code = seg.get('action_code', seg.get('camera_movement'))
    return CAMERA_DESCRIPTIONS.get(code, f'The camera {humanize_code(code)}.')


def other_segment_description(action):
    existing = action.get('one_sentence') or action.get('generated_sentence') or action.get('original_sentence')
    if is_set(existing):
        return str(existing)
    code = action.get('action_code')
    if code == 'ICG_SWITCH':
        return 'The ICG is switched.'
    tool = humanize_code(action.get('tool_type'))
    verb = ACTION_VERBS.get(code, humanize_code(code) or 'acts')
    target = humanize_code(action.get('target_structure'))
    subject = f'The {tool}' if tool else 'The other action'
    return f'{subject} {verb}{(" the " + target) if target else ""}.'


def naturalize_simple_segment(actor, seg):
    out = dict(seg)
    if actor == 'left':
        out['retraction_direction_code'] = humanize_code(seg.get('retraction_direction_code'))
        out['start_direction'] = humanize_code(seg.get('start_direction'))
        out['end_direction'] = humanize_code(seg.get('end_direction'))
        out['changed'] = {'yes': 'changed', 'no': 'not changed'}.get(seg.get('changed'), humanize_code(seg.get('changed')))
    elif actor == 'right':
        out['target_structure'] = humanize_code(seg.get('target_structure'))
        out['target_context_1'] = humanize_code(seg.get('target_context_1'))
        out['target_context_2'] = humanize_code(seg.get('target_context_2'))
        out['tool_type'] = humanize_code(seg.get('tool_type'))
        out['action_code'] = humanize_code(seg.get('action_code'))
        out['triplet'] = [humanize_code(value) for value in seg.get('triplet', [])]
    elif actor == 'camera':
        out['action_code'] = humanize_code(seg.get('action_code'))
        out['camera_movement'] = humanize_code(seg.get('camera_movement'))
    elif actor == 'other':
        for key in ['target_structure', 'target_context_1', 'target_context_2', 'actor_role', 'tool_type', 'action_code']:
            out[key] = humanize_code(seg.get(key))
    out['description'] = complete_sentence(seg.get('description'))
    return out


def naturalize_simple_actions(record):
    out = {key: value for key, value in record.items() if key not in ['left', 'right', 'camera', 'other']}
    for actor in ['left', 'right', 'camera', 'other']:
        out[actor] = [naturalize_simple_segment(actor, seg) for seg in record.get(actor, [])]
    return out


def convert_record(record):
    coarse = record.get('coarse', {})
    out = {
        'example_id': record.get('example_id'),
        'video_id': record.get('video_id'),
        'criterion': record.get('criterion'),
        'mind_change': record.get('mind_change'),
        'frame_range': [frame_or_none(coarse.get('start_frame')), frame_or_none(coarse.get('end_frame'))],
        'source_path': record.get('_source_path'),
        'left': [],
        'right': [],
        'camera': [],
        'other': [],
    }
    for action in coarse.get('annotation', {}).get('actions_ranked', []):
        rank = action.get('rank')
        actor_role = action.get('actor_role', '(not set)')
        handled = False

        for seg in action.get('left_action_segments') or []:
            changed, start_direction, end_direction = left_direction_fields(seg)
            left_row = {
                'start_frame': frame_or_none(seg.get('start_frame')),
                'end_frame': frame_or_none(seg.get('end_frame')),
                'retraction_direction_code': seg.get('action_code', '(not set)'),
                'changed': changed,
                'start_direction': start_direction,
                'end_direction': end_direction,
                'description': '',
                'rank': rank,
            }
            left_row['description'] = left_segment_description({**seg, **left_row, 'action_code': left_row['retraction_direction_code']})
            out['left'].append(left_row)
            handled = True

        for seg in action.get('right_action_segments') or []:
            triplet = [seg.get('tool_type', '(not set)'), seg.get('action_code', '(not set)'), seg.get('target_structure', '(not set)')]
            right_row = {
                'start_frame': frame_or_none(seg.get('start_frame')),
                'end_frame': frame_or_none(seg.get('end_frame')),
                'target_structure': triplet[2],
                'target_context_1': seg.get('target_context_1', '(not set)'),
                'target_context_2': seg.get('target_context_2', '(not set)'),
                'tool_type': triplet[0],
                'action_code': triplet[1],
                'triplet': triplet,
                'description': '',
                'rank': rank,
            }
            right_row['description'] = right_segment_description({**seg, **right_row})
            out['right'].append(right_row)
            handled = True

        for seg in action.get('camera_action_segments') or []:
            camera_row = {
                'start_frame': frame_or_none(seg.get('start_frame')),
                'end_frame': frame_or_none(seg.get('end_frame')),
                'action_code': seg.get('action_code', seg.get('camera_movement', '(not set)')),
                'camera_movement': seg.get('camera_movement', seg.get('action_code', '(not set)')),
                'description': '',
                'rank': rank,
            }
            camera_row['description'] = camera_segment_description({**seg, **camera_row})
            out['camera'].append(camera_row)
            handled = True

        action_code = action.get('action_code')
        if (not handled and actor_role not in {'left_instrument', 'right_instrument', 'camera'}) or action_code == 'ICG_SWITCH':
            out['other'].append(generic_other_segment(action))

    for key in ['left', 'right', 'camera', 'other']:
        out[key] = sorted_segments(out[key])
    return out

simple_records = [convert_record(record) for record in records]
natural_simple_records = [naturalize_simple_actions(record) for record in simple_records]
print('converted:', len(simple_records))
print(json.dumps(simple_records[0], indent=2)[:2000])


converted: 90
{
  "example_id": "00467596-8200-449c-8528-d4816ec2f6a2__C1__avg__c_001950_002400",
  "video_id": "00467596-8200-449c-8528-d4816ec2f6a2",
  "criterion": "C1",
  "mind_change": "unsatisfied->satisfied",
  "frame_range": [
    1950,
    2400
  ],
  "source_path": "/shared_data0/weiqiuy/surgent/data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1/audit_v11/00467596-8200-449c-8528-d4816ec2f6a2.json",
  "left": [
    {
      "start_frame": 2040,
      "end_frame": 2160,
      "retraction_direction_code": "RETRACT_MEDIAL_TO_LATERAL",
      "changed": "yes",
      "start_direction": "medial",
      "end_direction": "lateral",
      "description": "The grasper retracts the gallbladder neck from medial to lateral.",
      "rank": 2
    }
  ],
  "right": [
    {
      "start_frame": 1950,
      "end_frame": 2040,
      "target_structure": "CysticArtery",
      "target_context_1": "between cystic artery and cystic plate",
      "target_context_2": "close to the gallbladder n

In [4]:

def unique_sorted(values):
    return sorted({value for value in values if is_set(value)})

observed_left_retraction_direction_codes = unique_sorted(
    seg['retraction_direction_code']
    for record in simple_records
    for seg in record['left']
)
observed_right_triplets = sorted({
    tuple(seg['triplet'])
    for record in simple_records
    for seg in record['right']
    if all(is_set(value) for value in seg['triplet'])
})
observed_camera_action_codes = unique_sorted(
    seg['action_code']
    for record in simple_records
    for seg in record['camera']
)
observed_other_action_codes = unique_sorted(
    seg['action_code']
    for record in simple_records
    for seg in record['other']
)

simple_options = {
    'source': 'audit_timestamped_tool_interface.py + taxonomy_v10.json + observed audit_v11 GT',
    'interface_options': {
        'retraction_direction_options': RETRACTION_DIRECTION_OPTIONS,
        'camera_action_codes': CAMERA_ACTION_CODES,
        'tool_type': options['tool_type'],
        'action_code': options['action_code'],
        'target_structure': options['target_structure'],
        'target_context': options['target_context'],
    },
    'left_retraction_direction_code_options': observed_left_retraction_direction_codes,
    'right_triplet_options': [list(triplet) for triplet in observed_right_triplets],
    'camera_action_code_options': [code for code in CAMERA_ACTION_CODES if code in set(observed_camera_action_codes)] + [code for code in observed_camera_action_codes if code not in CAMERA_ACTION_CODES],
    'other_action_code_options': observed_other_action_codes,
}

natural_simple_options = {
    'source': simple_options['source'] + ' + deterministic natural-language transcription',
    'interface_options': {
        'retraction_direction_options': [humanize_code(value) for value in RETRACTION_DIRECTION_OPTIONS],
        'camera_action_codes': [humanize_code(value) for value in CAMERA_ACTION_CODES],
        'tool_type': [humanize_code(value) for value in options['tool_type']],
        'action_code': [humanize_code(value) for value in options['action_code']],
        'target_structure': [humanize_code(value) for value in options['target_structure']],
        'target_context': [humanize_code(value) for value in options['target_context']],
    },
    'left_retraction_direction_code_options': [humanize_code(value) for value in observed_left_retraction_direction_codes],
    'right_triplet_options': [[humanize_code(value) for value in triplet] for triplet in observed_right_triplets],
    'camera_action_code_options': [humanize_code(value) for value in simple_options['camera_action_code_options']],
    'other_action_code_options': [humanize_code(value) for value in observed_other_action_codes],
}

translation_values = set()
for record in simple_records:
    for seg in record['left']:
        translation_values.update([seg.get('retraction_direction_code'), seg.get('changed'), seg.get('start_direction'), seg.get('end_direction')])
    for seg in record['right']:
        translation_values.update([seg.get('tool_type'), seg.get('action_code'), seg.get('target_structure'), seg.get('target_context_1'), seg.get('target_context_2')])
    for seg in record['camera']:
        translation_values.update([seg.get('action_code'), seg.get('camera_movement')])
    for seg in record['other']:
        translation_values.update([seg.get('actor_role'), seg.get('tool_type'), seg.get('action_code'), seg.get('target_structure'), seg.get('target_context_1'), seg.get('target_context_2')])
translation_pairs = [
    {'code': value, 'natural_language': humanize_code(value)}
    for value in sorted(value for value in translation_values if is_set(value))
]

write_json(SIMPLE_GT_PATH, simple_records)
write_json(NATURAL_GT_PATH, natural_simple_records)
write_json(OPTIONS_PATH, simple_options)
write_json(NATURAL_OPTIONS_PATH, natural_simple_options)
write_json(TRANSLATION_PAIRS_PATH, translation_pairs)
print('left options:', simple_options['left_retraction_direction_code_options'])
print('right triplets:', len(simple_options['right_triplet_options']))
print('camera options:', simple_options['camera_action_code_options'])
print('other options:', simple_options['other_action_code_options'])
print('wrote', SIMPLE_GT_PATH)
print('wrote', NATURAL_GT_PATH)
print('wrote', OPTIONS_PATH)
print('wrote', NATURAL_OPTIONS_PATH)
print('wrote', TRANSLATION_PAIRS_PATH)


left options: ['KEEP_RETRACT_LATERAL', 'KEEP_RETRACT_MEDIAL', 'KEEP_RETRACT_UPWARD', 'RETRACT_LATERAL', 'RETRACT_LATERAL_TO_MEDIAL', 'RETRACT_LATERAL_TO_UPWARD', 'RETRACT_MEDIAL', 'RETRACT_MEDIAL_TO_LATERAL', 'RETRACT_UPWARD_TO_LATERAL']
right triplets: 21
camera options: ['CAMERA_ZOOM_IN', 'CAMERA_ZOOM_OUT', 'CAMERA_REPOSITION', 'CAMERA_UNCERTAIN', 'CAMERA_NO_CHANGE']
other options: ['ICG_SWITCH']
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_actions.json
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_actions_natural_language.json
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_action_options.json
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_action_options_natural_language.json
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_code_

In [5]:
def flatten_records(records_to_flatten):
    rows = []
    for record in records_to_flatten:
        base = {k: record[k] for k in ['example_id', 'video_id', 'criterion', 'mind_change']}
        base['clip_start_frame'], base['clip_end_frame'] = record['frame_range']
        for actor in ['left', 'right', 'camera', 'other']:
            for seg in record[actor]:
                row = dict(base)
                row['actor'] = actor
                row.update({k: json.dumps(v) if isinstance(v, list) else v for k, v in seg.items()})
                rows.append(row)
    return rows


def write_flat_csv(path, rows):
    fieldnames = sorted({key for row in rows for key in row})
    with open(path, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


flat_rows = flatten_records(simple_records)
natural_flat_rows = flatten_records(natural_simple_records)
write_flat_csv(CSV_PATH, flat_rows)
write_flat_csv(NATURAL_CSV_PATH, natural_flat_rows)
print('flat rows:', len(flat_rows))
print('natural flat rows:', len(natural_flat_rows))
print('wrote', CSV_PATH)
print('wrote', NATURAL_CSV_PATH)


flat rows: 285
natural flat rows: 285
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_actions_flat.csv
wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_actions_natural_language_flat.csv


In [ ]:
from IPython.display import HTML, IFrame, display
import base64
import html as html_lib
import io

FRAMES_DIR = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/frames/test'
VIS_PATH = OUT_DIR / 'audit_v11_simple_action_gt_visualization.html'
NATURAL_VIS_PATH = OUT_DIR / 'audit_v11_simple_action_gt_visualization_natural_language.html'

records_by_id = {record['example_id']: record for record in records}
actor_order = ['left', 'right', 'camera', 'other']
actor_labels = {
    'left': 'Left retraction',
    'right': 'Right dissection',
    'camera': 'Camera',
    'other': 'Other',
}
actor_colors = {
    'left': '#2563eb',
    'right': '#dc2626',
    'camera': '#7c3aed',
    'other': '#059669',
}


def esc(value):
    return html_lib.escape(str(value if value is not None else ''))


def frame_path(video_id, frame_id):
    frame_dir = FRAMES_DIR / str(video_id)
    for suffix in ['png', 'jpg', 'jpeg']:
        path = frame_dir / f'frame_{int(frame_id):06d}.{suffix}'
        if path.exists():
            return path
    if frame_dir.exists():
        existing = sorted(frame_dir.glob('frame_*.*'))
        if existing:
            def frame_number(path):
                try:
                    return int(path.stem.split('_')[-1])
                except Exception:
                    return 10**12
            return min(existing, key=lambda path: abs(frame_number(path) - int(frame_id)))
    return None


def image_data_uri(path, max_size=(360, 220), quality=72):
    if path is None or not path.exists():
        return None
    try:
        from PIL import Image
        with Image.open(path) as img:
            img.thumbnail(max_size)
            if img.mode not in {'RGB', 'L'}:
                img = img.convert('RGB')
            buf = io.BytesIO()
            img.save(buf, format='JPEG', quality=quality, optimize=True)
            data = base64.b64encode(buf.getvalue()).decode('ascii')
            return f'data:image/jpeg;base64,{data}'
    except Exception:
        suffix = path.suffix.lower().lstrip('.') or 'png'
        mime = 'jpeg' if suffix in {'jpg', 'jpeg'} else suffix
        data = base64.b64encode(path.read_bytes()).decode('ascii')
        return f'data:image/{mime};base64,{data}'


def review_frames(record, simple_record):
    coarse = record.get('coarse', {}) if record else {}
    start = int(simple_record['frame_range'][0])
    end = int(simple_record['frame_range'][1])
    mid = int(coarse.get('mid_frame', (start + end) // 2))
    frames = []
    seen = set()
    for label, frame_id in [('start', start), ('mid', mid), ('end', end)]:
        if frame_id in seen:
            continue
        seen.add(frame_id)
        frames.append((label, frame_id))
    return frames


def frames_html(record, simple_record):
    parts = []
    for label, frame_id in review_frames(record, simple_record):
        path = frame_path(simple_record['video_id'], frame_id)
        uri = image_data_uri(path)
        if uri:
            parts.append(
                '<figure class="frame-card">'
                f'<img src="{uri}" alt="{esc(label)} frame {int(frame_id)}">'
                f'<figcaption>{esc(label)} &middot; frame {int(frame_id)}</figcaption>'
                '</figure>'
            )
        else:
            parts.append(
                '<figure class="frame-card missing">'
                f'<div>Missing frame {int(frame_id)}</div>'
                f'<figcaption>{esc(label)}</figcaption>'
                '</figure>'
            )
    return '<div class="frames">' + ''.join(parts) + '</div>'


def segment_label(actor, seg):
    if actor == 'left':
        return seg.get('retraction_direction_code', '')
    if actor == 'right':
        return ' | '.join(str(seg.get(key, '')) for key in ['tool_type', 'action_code', 'target_structure'])
    if actor == 'camera':
        return seg.get('action_code', '')
    return seg.get('action_code', '')


def segment_tooltip(seg):
    return esc(json.dumps(seg, ensure_ascii=False))


def timeline_html(simple_record):
    clip_start, clip_end = [int(x) for x in simple_record['frame_range']]
    span = max(1, clip_end - clip_start)
    chunks = [
        '<div class="timeline">',
        f'<div class="scale"><span>{clip_start}</span><span>{clip_end}</span></div>',
    ]
    for actor in actor_order:
        color = actor_colors[actor]
        chunks.append(f'<div class="actor-row"><div class="actor-name">{esc(actor_labels[actor])}</div><div class="track">')
        for seg in simple_record.get(actor, []):
            start = int(seg.get('start_frame') or clip_start)
            end = int(seg.get('end_frame') or start)
            left = max(0, min(100, ((start - clip_start) / span) * 100))
            width = max(1.2, min(100 - left, ((end - start) / span) * 100))
            label = esc(segment_label(actor, seg))
            title = segment_tooltip(seg)
            chunks.append(
                f'<div class="bar" title="{title}" style="left:{left:.2f}%;width:{width:.2f}%;background:{color};">'
                f'<span>{label}</span></div>'
            )
        chunks.append('</div></div>')
    chunks.append('</div>')
    return ''.join(chunks)


def segments_table_html(simple_record):
    rows = []
    for actor in actor_order:
        for seg in simple_record.get(actor, []):
            rows.append(
                '<tr>'
                f'<td>{esc(actor)}</td>'
                f'<td>{esc(seg.get("start_frame"))}</td>'
                f'<td>{esc(seg.get("end_frame"))}</td>'
                f'<td>{esc(segment_label(actor, seg))}</td>'
                f'<td>{esc(seg.get("description", ""))}</td>'
                f'<td><code>{esc(json.dumps(seg, ensure_ascii=False))}</code></td>'
                '</tr>'
            )
    if not rows:
        return '<p class="empty"><em>No GT segments.</em></p>'
    return (
        '<table class="segments"><thead><tr>'
        '<th>actor</th><th>start</th><th>end</th><th>label</th><th>description</th><th>full row</th>'
        '</tr></thead><tbody>' + ''.join(rows) + '</tbody></table>'
    )


def record_html(simple_record, index):
    record = records_by_id.get(simple_record['example_id'])
    counts = ', '.join(f'{actor}:{len(simple_record.get(actor, []))}' for actor in actor_order)
    return (
        f'<details class="clip" {"open" if index < 3 else ""}>'
        '<summary>'
        f'<span class="clip-title">{index + 1:02d}. {esc(simple_record["criterion"])} &middot; {esc(simple_record["example_id"])}</span>'
        f'<span class="clip-meta">frames {esc(simple_record["frame_range"][0])}-{esc(simple_record["frame_range"][1])} &middot; {esc(counts)}</span>'
        '</summary>'
        f'{frames_html(record, simple_record)}'
        f'{timeline_html(simple_record)}'
        f'{segments_table_html(simple_record)}'
        '</details>'
    )


style = '''
<style>
  body {font-family: system-ui, -apple-system, BlinkMacSystemFont, "Segoe UI", sans-serif; margin: 0; color: #172033; background: #f6f8fb;}
  .page {max-width: 1320px; margin: 0 auto; padding: 18px 20px 32px;}
  h1 {font-size: 24px; margin: 0 0 6px;}
  .subtle {color: #5f6b7a; margin: 0 0 16px; font-size: 13px;}
  .legend {display: flex; flex-wrap: wrap; gap: 10px 18px; margin: 10px 0 18px; font-size: 13px;}
  .legend span:before {content: ''; display: inline-block; width: 10px; height: 10px; border-radius: 2px; margin-right: 6px; vertical-align: -1px; background: var(--color);}
  .clip {background: #fff; border: 1px solid #d9e0e8; border-radius: 8px; margin: 12px 0; overflow: hidden;}
  .clip summary {cursor: pointer; display: flex; flex-wrap: wrap; align-items: center; justify-content: space-between; gap: 8px; padding: 10px 12px; background: #fdfefe; border-bottom: 1px solid #e7ecf2;}
  .clip-title {font-weight: 650; font-size: 14px;}
  .clip-meta {color: #5f6b7a; font-size: 12px;}
  .frames {display: grid; grid-template-columns: repeat(auto-fit, minmax(210px, 1fr)); gap: 10px; padding: 12px;}
  .frame-card {margin: 0; border: 1px solid #e1e7ef; border-radius: 6px; overflow: hidden; background: #0f172a;}
  .frame-card img {display: block; width: 100%; height: 160px; object-fit: contain;}
  .frame-card figcaption {background: #fff; color: #526070; font-size: 12px; padding: 5px 7px;}
  .frame-card.missing div {height: 160px; display: flex; align-items: center; justify-content: center; color: #94a3b8; background: #111827;}
  .timeline {padding: 4px 12px 12px;}
  .scale {display: flex; justify-content: space-between; color: #64748b; font-size: 11px; margin: 0 0 3px 144px;}
  .actor-row {display: grid; grid-template-columns: 136px 1fr; align-items: center; gap: 8px; margin: 7px 0;}
  .actor-name {font-size: 12px; color: #344054; text-align: right;}
  .track {position: relative; height: 28px; background: #eef2f6; border: 1px solid #d9e0e8; border-radius: 5px; overflow: hidden;}
  .bar {position: absolute; top: 3px; bottom: 3px; border-radius: 4px; color: #fff; font-size: 11px; line-height: 20px; overflow: hidden; white-space: nowrap; text-overflow: ellipsis; padding: 0 5px; box-sizing: border-box;}
  .bar span {text-shadow: 0 1px 1px rgba(0,0,0,.35);}
  table.segments {width: calc(100% - 24px); margin: 0 12px 12px; border-collapse: collapse; font-size: 12px;}
  table.segments th, table.segments td {border: 1px solid #e1e7ef; padding: 5px 7px; vertical-align: top; text-align: left;}
  table.segments th {background: #f8fafc; color: #344054;}
  table.segments code {white-space: pre-wrap; overflow-wrap: anywhere; color: #344054;}
  .empty {margin: 0 12px 12px; color: #5f6b7a;}
</style>
'''
def render_visualization(records_to_render, output_path, title):
    legend = ''.join(
        f'<span style="--color:{actor_colors[actor]}">{esc(actor_labels[actor])}</span>'
        for actor in actor_order
    )
    body = ''.join(record_html(record, index) for index, record in enumerate(records_to_render))
    html_doc = (
        '<!doctype html><html><head><meta charset="utf-8"><title>' + esc(title) + '</title>'
        + style
        + '</head><body><main class="page">'
        + f'<h1>{esc(title)}</h1>'
        + f'<p class="subtle">{len(records_to_render)} clips converted from timestamped audit annotations. First three clips are expanded by default.</p>'
        + f'<div class="legend">{legend}</div>'
        + body
        + '</main></body></html>'
    )
    output_path.write_text(html_doc)
    print('wrote', output_path)


render_visualization(simple_records, VIS_PATH, 'audit_v11 simple action GT visualization - code labels')
render_visualization(natural_simple_records, NATURAL_VIS_PATH, 'audit_v11 simple action GT visualization - natural language labels')
display(HTML(f'<p><b>Code GT visualization:</b> <code>{esc(VIS_PATH)}</code><br><b>Natural-language GT visualization:</b> <code>{esc(NATURAL_VIS_PATH)}</code></p>'))
display(IFrame(src=str(VIS_PATH.relative_to(ROOT_DIR / 'notebooks')), width='100%', height=520))


wrote /shared_data0/weiqiuy/surgent/notebooks/artifacts/audit_v11_simple_action_gt/audit_v11_simple_action_gt_visualization.html
